In [1]:
import json
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

certify_root = repo_root / "output" / "ner_conll2003_bert" / "certify" / "last" / "euclidean" / "annoy" / "annoy_index.ann"

In [2]:
def load_results(sigma_folder):
    """Load results from a sigma folder (completed or running)."""
    path = certify_root / sigma_folder
    
    # Try completed metrics first
    metrics_path = path / "metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            data = json.load(f)
        return {"status": "completed", "data": data}
    
    # Fall back to running metrics
    running_path = path / "running_metrics.json"
    if running_path.exists():
        with open(running_path) as f:
            data = json.load(f)
        return {"status": "running", "data": data}
    
    return None

# Find all sigma folders
sigma_folders = sorted([d.name for d in certify_root.iterdir() if d.is_dir() and d.name.startswith("sigma")])
print(f"Found sigma folders: {sigma_folders}")

Found sigma folders: ['sigma_0_20', 'sigma_0_50']


## Overall Results by Sigma

In [3]:
results = []
for folder in sigma_folders:
    r = load_results(folder)
    if r:
        sigma = folder.replace("sigma_", "").replace("_", ".")
        if r["status"] == "completed":
            d = r["data"]
            results.append({
                "sigma": sigma,
                "status": "✓ completed",
                "clean_f1": f"{d['clean']['f1']:.2%}",
                "smooth_f1": f"{d['smoothed']['f1']:.2%}",
                "clean_acc": f"{d['clean']['token_acc']:.2%}",
                "certified_acc": f"{d['smoothed']['certified_token_acc']:.2%}",
                "mean_radius": f"{d['smoothed']['mean_certified_radius']:.3f}",
                "abstention": f"{d['smoothed']['abstention_rate']:.4%}",
                "sentences": d['smoothed']['sentences_evaluated'],
            })
        else:
            d = r["data"]["running"]
            results.append({
                "sigma": sigma,
                "status": f"⏳ batch {r['data']['last_completed_batch']}",
                "clean_f1": "-",
                "smooth_f1": f"{d['f1']:.2%}",
                "clean_acc": "-",
                "certified_acc": f"{d['certified_token_acc']:.2%}",
                "mean_radius": f"{d['mean_certified_radius']:.3f}",
                "abstention": f"{d['abstention_rate']:.4%}",
                "sentences": d['sentences_evaluated'],
            })

df = pd.DataFrame(results)
print("=" * 80)
print("CERTIFICATION RESULTS SUMMARY")
print("=" * 80)
print(df.to_string(index=False))

CERTIFICATION RESULTS SUMMARY
sigma      status clean_f1 smooth_f1 clean_acc certified_acc mean_radius abstention  sentences
 0.20 ✓ completed   88.77%    88.70%    97.94%        97.91%       0.300    0.0081%       2400
 0.50  ⏳ batch 39        -    92.41%         -        97.47%       0.749    0.0145%        624


## Detailed Results: σ = 0.20 (Completed)

In [4]:
# Load σ=0.20 results
r020 = load_results("sigma_0_20")
if r020 and r020["status"] == "completed":
    d = r020["data"]
    
    print("CLEAN BASELINE:")
    print(f"  F1: {d['clean']['f1']:.2%}")
    print(f"  Precision: {d['clean']['precision']:.2%}")
    print(f"  Recall: {d['clean']['recall']:.2%}")
    print(f"  Token Acc: {d['clean']['token_acc']:.2%}")
    
    print("\nSMOOTHED (Certified):")
    print(f"  F1: {d['smoothed']['f1']:.2%}")
    print(f"  Precision: {d['smoothed']['precision']:.2%}")
    print(f"  Recall: {d['smoothed']['recall']:.2%}")
    print(f"  Token Acc: {d['smoothed']['token_acc']:.2%}")
    print(f"  Certified Acc: {d['smoothed']['certified_token_acc']:.2%}")
    print(f"  Mean Radius: {d['smoothed']['mean_certified_radius']:.4f}")
    print(f"  Abstention Rate: {d['smoothed']['abstention_rate']:.4%}")
    print(f"  Total Tokens: {d['smoothed']['total_certified_tokens']:,}")
    print(f"  Certified Correct: {d['smoothed']['certified_correct_tokens']:,}")

CLEAN BASELINE:
  F1: 88.77%
  Precision: 88.28%
  Recall: 89.26%
  Token Acc: 97.94%

SMOOTHED (Certified):
  F1: 88.70%
  Precision: 88.13%
  Recall: 89.29%
  Token Acc: 97.91%
  Certified Acc: 97.91%
  Mean Radius: 0.3000
  Abstention Rate: 0.0081%
  Total Tokens: 36,978
  Certified Correct: 36,205


## Certification by Entity Type

In [5]:
if r020 and r020["status"] == "completed":
    cert_by_label = r020["data"]["smoothed"].get("certification_by_label", {})
    
    # Group by entity type
    entity_stats = {}
    for label, stats in cert_by_label.items():
        if label == "O":
            entity = "O"
        else:
            entity = label.split("-")[1] if "-" in label else label
        
        if entity not in entity_stats:
            entity_stats[entity] = {"total": 0, "correct": 0, "abstained": 0}
        entity_stats[entity]["total"] += stats["total_tokens"]
        entity_stats[entity]["correct"] += stats["certified_correct_tokens"]
        entity_stats[entity]["abstained"] += stats["abstained_tokens"]
    
    rows = []
    for entity in ["O", "PER", "ORG", "LOC", "MISC"]:
        if entity in entity_stats:
            s = entity_stats[entity]
            rows.append({
                "Entity": entity,
                "Total": s["total"],
                "Correct": s["correct"],
                "Cert Acc": f"{s['correct']/s['total']:.2%}" if s['total'] else "-",
                "Abstained": s["abstained"],
            })
    
    df_entity = pd.DataFrame(rows)
    print("CERTIFICATION BY ENTITY TYPE (σ=0.20):")
    print(df_entity.to_string(index=False))

CERTIFICATION BY ENTITY TYPE (σ=0.20):
Entity  Total  Correct Cert Acc  Abstained
     O  31331    31070   99.17%          2
   PER   1789     1716   95.92%          0
   ORG   1712     1502   87.73%          1
   LOC   1432     1336   93.30%          0
  MISC    714      581   81.37%          0


## Certification by Label (B-/I- breakdown)

In [6]:
if r020 and r020["status"] == "completed":
    cert_by_label = r020["data"]["smoothed"].get("certification_by_label", {})
    
    rows = []
    label_order = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
    for label in label_order:
        if label in cert_by_label:
            s = cert_by_label[label]
            rows.append({
                "Label": label,
                "Total": s["total_tokens"],
                "Certified": s["certified_tokens"],
                "Correct": s["certified_correct_tokens"],
                "Cert Acc": f"{s['certified_accuracy']:.2%}",
                "Abstained": s["abstained_tokens"],
            })
    
    df_label = pd.DataFrame(rows)
    print("CERTIFICATION BY LABEL (σ=0.20):")
    print(df_label.to_string(index=False))

CERTIFICATION BY LABEL (σ=0.20):
 Label  Total  Certified  Correct Cert Acc  Abstained
     O  31331      31329    31070   99.17%          2
 B-PER   1088       1088     1028   94.49%          0
 I-PER    701        701      688   98.15%          0
 B-ORG   1062       1062      918   86.44%          0
 I-ORG    650        649      584   89.98%          1
 B-LOC   1227       1227     1153   93.97%          0
 I-LOC    205        205      183   89.27%          0
B-MISC    572        572      479   83.74%          0
I-MISC    142        142      102   71.83%          0


---
# Masked Certification Results for context masking

In [19]:
masked_certify_root = repo_root / "output" / "ner_conll2003_bert" / "masked_certify" / "last" / "euclidean" / "annoy" / "annoy_index.ann" / "context"

def load_masked_results(sigma_folder):
    """Load results from a masked sigma folder (completed or running)."""
    path = masked_certify_root / sigma_folder
    
    # Try results_summary.json first (new format)
    summary_path = path / "results_summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            data = json.load(f)
        return {"status": "completed", "format": "summary", "data": data}
    
    # Try completed metrics.json
    metrics_path = path / "metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            data = json.load(f)
        return {"status": "completed", "format": "metrics", "data": data}
    
    # Fall back to running metrics
    running_path = path / "running_metrics.json"
    if running_path.exists():
        with open(running_path) as f:
            data = json.load(f)
        return {"status": "running", "format": "running", "data": data}
    
    return None

# Find all masked sigma folders
if masked_certify_root.exists():
    masked_sigma_folders = sorted([d.name for d in masked_certify_root.iterdir() if d.is_dir() and d.name.startswith("sigma")])
    print(f"Found masked sigma folders: {masked_sigma_folders}")
else:
    masked_sigma_folders = []
    print(f"Masked certify folder not found: {masked_certify_root}")

Found masked sigma folders: ['sigma_0_20', 'sigma_0_50']


## Masked Overall Results by Sigma

In [20]:
masked_results = []
for folder in masked_sigma_folders:
    r = load_masked_results(folder)
    if r:
        sigma = folder.replace("sigma_", "").replace("_", ".")
        if r["status"] == "completed" and r.get("format") == "summary":
            # New results_summary.json format
            d = r["data"]
            overall = d.get("overall", {})
            cert = d.get("certification_totals", {})
            masked_results.append({
                "sigma": sigma,
                "status": "✓ completed",
                "f1": f"{overall.get('f1', 0):.2%}",
                "token_acc": f"{overall.get('token_accuracy', 0):.2%}",
                "certified_acc": f"{cert.get('certified_correct_tokens', 0) / cert.get('total_certified_tokens', 1):.2%}",
                "mean_radius": f"{cert.get('mean_certified_radius', 0):.3f}",
                "abstention": f"{cert.get('abstention_rate', 0):.4%}",
                "tokens": cert.get('total_certified_tokens', 0),
            })
        elif r["status"] == "completed":
            # Old metrics.json format
            d = r["data"]
            masked_results.append({
                "sigma": sigma,
                "status": "✓ completed",
                "f1": f"{d['clean']['f1']:.2%}",
                "token_acc": f"{d['clean']['token_acc']:.2%}",
                "certified_acc": f"{d['smoothed']['certified_token_acc']:.2%}",
                "mean_radius": f"{d['smoothed']['mean_certified_radius']:.3f}",
                "abstention": f"{d['smoothed']['abstention_rate']:.4%}",
                "tokens": d['smoothed'].get('num_tokens', 0),
            })
        else:
            # Running format
            d = r["data"]["running"]
            masked_results.append({
                "sigma": sigma,
                "status": f"⏳ batch {r['data']['last_completed_batch']}",
                "f1": f"{d['f1']:.2%}",
                "token_acc": "-",
                "certified_acc": f"{d['certified_token_acc']:.2%}",
                "mean_radius": f"{d['mean_certified_radius']:.3f}",
                "abstention": f"{d['abstention_rate']:.4%}",
                "tokens": d.get('num_tokens', 0),
            })

if masked_results:
    df_masked = pd.DataFrame(masked_results)
    print("=" * 80)
    print("MASKED CERTIFICATION RESULTS SUMMARY")
    print("=" * 80)
    print(df_masked.to_string(index=False))
else:
    print("No masked certification results found yet.")

MASKED CERTIFICATION RESULTS SUMMARY
sigma      status     f1 token_acc certified_acc mean_radius abstention  tokens
 0.20 ✓ completed 81.76%    96.45%        96.45%       0.299    0.0892%   36978
 0.50  ⏳ batch 52 85.71%         -        95.93%       0.744    0.1915%       0


## Masked Detailed Results

In [21]:
# Pick a sigma to examine (e.g., σ=0.20)
masked_sigma = "sigma_0_20"
masked_sigma_path = masked_certify_root / masked_sigma

if masked_sigma_path.exists():
    # Load results_summary.json
    summary_path = masked_sigma_path / "results_summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            masked_detail = json.load(f)
        
        overall = masked_detail.get("overall", {})
        cert = masked_detail.get("certification_totals", {})
        
        print(f"=== MASKED σ={masked_sigma.replace('sigma_', '').replace('_', '.')} ===\n")
        print(f"Experiment: {masked_detail.get('experiment_name', 'N/A')}")
        print(f"Split: {masked_detail.get('split', 'N/A')}")
        print(f"Total Tokens: {cert.get('total_certified_tokens', 'N/A')}")
        print()
        print("OVERALL PERFORMANCE:")
        print(f"  Precision: {overall.get('precision', 0)*100:.2f}%")
        print(f"  Recall: {overall.get('recall', 0)*100:.2f}%")
        print(f"  F1: {overall.get('f1', 0)*100:.2f}%")
        print(f"  Token Accuracy: {overall.get('token_accuracy', 0)*100:.2f}%")
        print()
        print("CERTIFICATION:")
        print(f"  Certified Correct: {cert.get('certified_correct_tokens', 0)} / {cert.get('total_certified_tokens', 0)}")
        print(f"  Certified Accuracy: {cert.get('certified_correct_tokens', 0) / max(cert.get('total_certified_tokens', 1), 1) * 100:.2f}%")
        print(f"  Mean Certified Radius: {cert.get('mean_certified_radius', 0):.4f}")
        print(f"  Abstention Rate: {cert.get('abstention_rate', 0)*100:.4f}%")
    else:
        print(f"results_summary.json not found in {masked_sigma_path}")
else:
    print(f"Masked folder not found: {masked_sigma_path}")

=== MASKED σ=0.20 ===

Experiment: ner_conll2003_bert_masking_certify
Split: test
Total Tokens: 36978

OVERALL PERFORMANCE:
  Precision: 81.23%
  Recall: 82.30%
  F1: 81.76%
  Token Accuracy: 96.45%

CERTIFICATION:
  Certified Correct: 35664 / 36978
  Certified Accuracy: 96.45%
  Mean Certified Radius: 0.2992
  Abstention Rate: 0.0892%


## Masked Certification by Entity Type

In [22]:
# Certification by entity type (O, PER, ORG, LOC, MISC)
cert_by_entity = masked_detail.get('certification_by_entity', {})
if cert_by_entity:
    entity_rows = []
    for entity, stats in cert_by_entity.items():
        entity_rows.append({
            'Entity': entity,
            'Tokens': stats.get('total_tokens', 0),
            'Certified': stats.get('certified_tokens', 0),
            'Correct': stats.get('certified_correct_tokens', 0),
            'Cert Acc': f"{stats.get('certified_accuracy', 0)*100:.2f}%",
            'Abstain': f"{stats.get('abstention_rate', 0)*100:.2f}%"
        })
    
    df_entity = pd.DataFrame(entity_rows)
    df_entity = df_entity.sort_values('Tokens', ascending=False)
    print(df_entity.to_string(index=False))
else:
    print("No certification_by_entity found in masked results")

Entity  Tokens  Certified  Correct Cert Acc Abstain
     O   31331      31307    30869   98.60%   0.08%
   PER    1789       1789     1687   94.30%   0.00%
   ORG    1712       1707     1334   78.15%   0.29%
   LOC    1432       1429     1249   87.40%   0.21%
  MISC     714        713      525   73.63%   0.14%


## Masked Certification by Label

In [23]:
# Full breakdown by label (B-PER, I-PER, B-ORG, etc.)
cert_by_label = masked_detail.get('certification_by_label', {})
if cert_by_label:
    label_rows = []
    for label, stats in cert_by_label.items():
        label_rows.append({
            'Label': label,
            'Tokens': stats.get('total_tokens', 0),
            'Certified': stats.get('certified_tokens', 0),
            'Correct': stats.get('certified_correct_tokens', 0),
            'Cert Acc': f"{stats.get('certified_accuracy', 0)*100:.2f}%",
            'Abstain': f"{stats.get('abstention_rate', 0)*100:.2f}%"
        })
    
    df_label = pd.DataFrame(label_rows)
    df_label = df_label.sort_values('Tokens', ascending=False)
    print(df_label.to_string(index=False))
else:
    print("No certification_by_label found in masked results")

 Label  Tokens  Certified  Correct Cert Acc Abstain
     O   31331      31307    30869   98.60%   0.08%
 B-LOC    1227       1225     1079   88.08%   0.16%
 B-PER    1088       1088     1011   92.92%   0.00%
 B-ORG    1062       1057      816   77.20%   0.47%
 I-PER     701        701      676   96.43%   0.00%
 I-ORG     650        650      518   79.69%   0.00%
B-MISC     572        572      433   75.70%   0.00%
 I-LOC     205        204      170   83.33%   0.49%
I-MISC     142        141       92   65.25%   0.70%
